## JSB Chorales (PyTorch)

Train an LSTM classifier to predict the **next 4-note vector** from the previous **50 timesteps**.


In [1]:
import os
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


def seed_all(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_all(42)

device = (
    torch.device("cuda")
    if torch.cuda.is_available()
    else torch.device("mps")
    if torch.backends.mps.is_available()
    else torch.device("cpu")
)

print("device:", device)
print("torch:", torch.__version__)


device: mps
torch: 2.11.0


In [2]:
DATA_DIR = Path("jsb_chorales")
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

train_files = sorted(TRAIN_DIR.glob("chorale_*.csv"))
test_files = sorted(TEST_DIR.glob("chorale_*.csv"))

print("#train files:", len(train_files))
print("#test files:", len(test_files))
print("example:", train_files[0])


def load_chorale_csv(path: Path) -> torch.Tensor:
    df = pd.read_csv(path)
    x = df[["note0", "note1", "note2", "note3"]].to_numpy(dtype=np.int64)
    return torch.from_numpy(x)  # (T, 4)


train_chorales = [load_chorale_csv(p) for p in train_files]
test_chorales = [load_chorale_csv(p) for p in test_files]

max_note = int(torch.max(torch.cat(train_chorales)).item())
min_note = int(torch.min(torch.cat(train_chorales)).item())
print("train note range:", (min_note, max_note))
print("example chorale shape:", train_chorales[0].shape)


def build_tuple_vocab(chorales: list[torch.Tensor]):
    tuples = set()
    for c in chorales:
        for row in c.tolist():
            tuples.add(tuple(int(v) for v in row))
    tuples = sorted(tuples)
    stoi = {t: i for i, t in enumerate(tuples)}
    return tuples, stoi


tuple_vocab, tuple_to_id = build_tuple_vocab(train_chorales)
UNK_ID = len(tuple_vocab)
num_classes = len(tuple_vocab) + 1
print("#classes (train unique 4-note vectors) + UNK:", num_classes)


class ChoraleSequenceDataset(Dataset):
    def __init__(
        self,
        chorales: list[torch.Tensor],
        seq_len: int,
        tuple_to_id: dict[tuple[int, int, int, int], int],
        unk_id: int,
    ):
        self.chorales = chorales
        self.seq_len = seq_len
        self.tuple_to_id = tuple_to_id
        self.unk_id = unk_id

        index_map: list[tuple[int, int]] = []  # (chorale_idx, start)
        for ci, c in enumerate(self.chorales):
            T = c.shape[0]
            for start in range(0, T - seq_len):
                index_map.append((ci, start))
        self.index_map = index_map

    def __len__(self):
        return len(self.index_map)

    def __getitem__(self, idx: int):
        ci, start = self.index_map[idx]
        c = self.chorales[ci]
        x = c[start : start + self.seq_len]  # (seq_len, 4)
        y_tuple = tuple(int(v) for v in c[start + self.seq_len].tolist())
        y = self.tuple_to_id.get(y_tuple, self.unk_id)
        return x, torch.tensor(y, dtype=torch.long)


SEQ_LEN = 50
train_ds = ChoraleSequenceDataset(train_chorales, SEQ_LEN, tuple_to_id, UNK_ID)
test_ds = ChoraleSequenceDataset(test_chorales, SEQ_LEN, tuple_to_id, UNK_ID)

print("train samples:", len(train_ds))
print("test samples:", len(test_ds))

# how many test targets are unseen in train?
unk_count = 0
for _, y in DataLoader(test_ds, batch_size=2048):
    unk_count += int((y == UNK_ID).sum().item())
print("test UNK targets:", unk_count, f"({unk_count/len(test_ds):.2%})")


#train files: 229
#test files: 77
example: jsb_chorales/train/chorale_000.csv
train note range: (0, 81)
example chorale shape: torch.Size([192, 4])
#classes (train unique 4-note vectors) + UNK: 4371
train samples: 43778
test samples: 15050
test UNK targets: 1740 (11.56%)


In [3]:
@dataclass
class TrainConfig:
    batch_size: int = 256
    epochs: int = 10
    lr: float = 3e-3
    clip_norm: float = 1.0
    emb_dim: int = 16
    hidden_size: int = 128
    num_layers: int = 2
    dropout: float = 0.1


class ChoraleNextVectorLSTM(nn.Module):
    def __init__(
        self,
        note_vocab_size: int,
        emb_dim: int,
        hidden_size: int,
        num_layers: int,
        num_classes: int,
        dropout: float,
    ):
        super().__init__()
        self.note_embed = nn.Embedding(note_vocab_size, emb_dim)
        self.lstm = nn.LSTM(
            input_size=4 * emb_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, 4) int64
        emb = self.note_embed(x)  # (B, T, 4, E)
        emb = emb.reshape(emb.shape[0], emb.shape[1], -1)  # (B, T, 4E)
        out, _ = self.lstm(emb)  # (B, T, H)
        last = out[:, -1, :]  # (B, H)
        logits = self.head(last)  # (B, C)
        return logits


cfg = TrainConfig()

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0)

model = ChoraleNextVectorLSTM(
    note_vocab_size=max_note + 1,
    emb_dim=cfg.emb_dim,
    hidden_size=cfg.hidden_size,
    num_layers=cfg.num_layers,
    num_classes=num_classes,
    dropout=cfg.dropout,
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)


def run_epoch(loader: DataLoader, train: bool):
    model.train(train)
    total_loss = 0.0
    correct = 0
    total = 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        if train:
            optimizer.zero_grad(set_to_none=True)

        logits = model(x)
        loss = criterion(logits, y)

        if train:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.clip_norm)
            optimizer.step()

        total_loss += float(loss.item()) * x.shape[0]
        pred = logits.argmax(dim=-1)
        correct += int((pred == y).sum().item())
        total += int(x.shape[0])

    return total_loss / max(total, 1), correct / max(total, 1)


for epoch in range(1, cfg.epochs + 1):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    test_loss, test_acc = run_epoch(test_loader, train=False)
    print(
        f"epoch {epoch:02d} | "
        f"train loss {train_loss:.4f} acc {train_acc:.4f} | "
        f"test loss {test_loss:.4f} acc {test_acc:.4f}"
    )

print("\nFinal test accuracy:", test_acc)


epoch 01 | train loss 5.2032 acc 0.2726 | test loss 4.7164 acc 0.3437
epoch 02 | train loss 3.2739 acc 0.4601 | test loss 4.2906 acc 0.4023
epoch 03 | train loss 2.5877 acc 0.5307 | test loss 4.1818 acc 0.4231
epoch 04 | train loss 2.2186 acc 0.5672 | test loss 4.1575 acc 0.4322
epoch 05 | train loss 1.9873 acc 0.5922 | test loss 4.1799 acc 0.4330
epoch 06 | train loss 1.8307 acc 0.6060 | test loss 4.2050 acc 0.4333
epoch 07 | train loss 1.7091 acc 0.6230 | test loss 4.2224 acc 0.4329
epoch 08 | train loss 1.6260 acc 0.6296 | test loss 4.2783 acc 0.4260
epoch 09 | train loss 1.5485 acc 0.6390 | test loss 4.3441 acc 0.4208
epoch 10 | train loss 1.4901 acc 0.6466 | test loss 4.3532 acc 0.4225

Final test accuracy: 0.4224584717607973
